# BNN MC Dropout - comparaison avec torchbnn

Variante du BNN multimodal où la tête bayésienne est remplacée par un MLP avec dropout maintenu actif à l'inférence (MC Dropout, Gal & Ghahramani 2016). Architecture, données et entraînement strictement identiques au notebook 03 pour une comparaison propre.

**Objectif :** justifier expérimentalement le choix de l'inférence variationnelle (torchbnn) plutôt que MC Dropout, en comparant performances, calibration et qualité de l'incertitude.

**Prérequis :** `runs_cnn_baseline/best_final.pt` (notebook 01), `predicted_clinical.pkl` (notebook 02).

**Sortie :** `runs_mc_dropout/best.pt` et tableaux comparatifs.

### Imports & seed

In [1]:
import os, glob, random, pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("mps" if torch.backends.mps.is_available() else
                      "cuda" if torch.cuda.is_available() else "cpu")
print(device)

mps


### Chemins & config

In [2]:
DATAROOT  = "/Users/teodul/Documents/Memoire/Epic and CSCR hospital Dataset"
TRAIN_DIR = os.path.join(DATAROOT, "Train")
TEST_DIR  = os.path.join(DATAROOT, "Test")
CNN_CKPT  = "runs_cnn_baseline/best_final.pt"
SAVE_DIR  = "runs_mc_dropout"
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS       = 15
LR           = 1e-3
DROPOUT_RATE = 0.3
MC_PASSES    = 50
CLASSES      = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES  = 4
PATIENCE     = 3

with open("predicted_clinical.pkl", "rb") as f:
    predicted_clinical = pickle.load(f)

### Architecture - MLP standard avec dropout

In [3]:
class MCDropoutMultimodalNet(nn.Module):
    def __init__(self, num_classes=4, clinical_dim=1, dropout=DROPOUT_RATE):
        super().__init__()

        backbone    = models.resnet18(weights="IMAGENET1K_V1")
        backbone.fc = nn.Identity()
        self.cnn    = backbone

        for name, param in self.cnn.named_parameters():
            if "layer4" in name or "layer3" in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.clinical_mlp = nn.Sequential(
            nn.Linear(clinical_dim, 64), nn.ReLU(),
            nn.Linear(64, 64),          nn.ReLU(),
            nn.Linear(64, 32),          nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(544, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, img, clinical):
        img_feat      = self.cnn(img)
        clinical_feat = self.clinical_mlp(clinical)
        fused         = torch.cat([img_feat, clinical_feat], dim=1)
        return self.head(fused)

    def enable_mc_dropout(self):
        """Active le dropout pour l'inférence MC."""
        for m in self.head.modules():
            if isinstance(m, nn.Dropout):
                m.train()


model = MCDropoutMultimodalNet().to(device)
ckpt_cnn = torch.load(CNN_CKPT, map_location=device, weights_only=False)
cnn_sd   = {k: v for k, v in ckpt_cnn["model_state"].items()
            if not k.startswith("fc.")}
missing, unexpected = model.cnn.load_state_dict(cnn_sd, strict=False)
print(f"CNN baseline chargé dans la branche image (missing={len(missing)}, unexpected={len(unexpected)})")

CNN baseline chargé dans la branche image (missing=0, unexpected=0)


### Dataset & dataloaders

In [ ]:
import re


class MultimodalBrainDataset(Dataset):
    def __init__(self, rootdir, transform, split=None, classes=CLASSES, samples=None):
        self.transform = transform
        self.classes = classes

        if samples is not None:
            self.samples = list(samples)
        else:
            self.samples = []
            for idx, cls in enumerate(classes):
                clsdir = os.path.join(rootdir, cls)
                for ext in ['.jpg', '.jpeg', '.png']:
                    for path in sorted(glob.glob(os.path.join(clsdir, f'*{ext}'))):
                        clin = predicted_clinical[split].get(path, np.array([0.5], dtype=np.float32))
                        self.samples.append((path, idx, cls, clin))
            random.shuffle(self.samples)

        counts = np.bincount([s[1] for s in self.samples])
        print(f"Dataset - {len(self.samples)} images")
        print("  " + ", ".join(f"{c}:{n}" for c, n in zip(classes, counts)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, clsname, clinical = self.samples[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        clinical = torch.tensor(clinical, dtype=torch.float32)
        return img, clinical, label


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])


def get_patient_id(filepath):
    """Extrait l'identifiant patient depuis le nom de fichier (identique au notebook 03)."""
    fname = os.path.basename(filepath)
    m = re.match(r'^(?:Tr|Te)-([a-z]{2}_\d+)', fname)
    if m: return m.group(1)
    m = re.match(r'^([A-Z]_\d+)_', fname)
    if m: return m.group(1)
    m = re.match(r'^([A-Z]_\d+)\.', fname)
    if m: return m.group(1)
    m = re.match(r'^(\d+)', fname)
    if m: return m.group(1)
    return fname


all_samples = []
for idx, cls in enumerate(CLASSES):
    clsdir = os.path.join(TRAIN_DIR, cls)
    for ext in ['.jpg', '.jpeg', '.png']:
        for path in sorted(glob.glob(os.path.join(clsdir, f'*{ext}'))):
            clin = predicted_clinical["train"].get(path, np.array([0.5], dtype=np.float32))
            all_samples.append((path, idx, cls, clin))

train_samples, val_samples = [], []
rng = np.random.default_rng(SEED)

for cls_idx, cls in enumerate(CLASSES):
    cls_samps = [(p, l, c, cl) for p, l, c, cl in all_samples if l == cls_idx]
    patient_ids = [get_patient_id(p) for p, _, _, _ in cls_samps]
    unique_patients = sorted(set(patient_ids))

    rng.shuffle(unique_patients)
    n_val_p = max(1, int(0.15 * len(unique_patients)))
    val_patients = set(unique_patients[:n_val_p])

    for s, pid in zip(cls_samps, patient_ids):
        (val_samples if pid in val_patients else train_samples).append(s)

random.shuffle(train_samples)

train_ds = MultimodalBrainDataset(None, train_tf, samples=train_samples)
val_ds   = MultimodalBrainDataset(None, val_tf,   samples=val_samples)
test_ds  = MultimodalBrainDataset(TEST_DIR, val_tf, split="test")

print(f"\nTrain: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

labels         = [s[1] for s in train_ds.samples]
counts         = np.bincount(labels)
class_weights  = 1.0 / counts
sample_weights = class_weights[labels]
g_gen = torch.Generator(); g_gen.manual_seed(SEED)
sampler = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.double),
    num_samples = len(sample_weights),
    replacement = True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, generator=g_gen)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

Dataset — 8103 images
  glioma:2537, meningioma:1822, notumor:1648, pituitary:2096
Dataset — 1541 images
  glioma:481, meningioma:361, notumor:291, pituitary:408
Dataset — 2414 images
  glioma:755, meningioma:546, notumor:487, pituitary:626

Train: 8103 | Val: 1541 | Test: 2414


### Entraînement

In [5]:
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable, lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
criterion = nn.CrossEntropyLoss()

best_auc, patience_count = 0.0, 0
history = {"train_loss": [], "val_loss": [], "val_auc": []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for imgs, clins, labels in train_loader:
        imgs, clins, labels = imgs.to(device), clins.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs, clins)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, clins, labels in val_loader:
            imgs, clins, labels = imgs.to(device), clins.to(device), labels.to(device)
            logits = model(imgs, clins)
            val_loss += criterion(logits, labels).item() * imgs.size(0)
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    val_loss /= len(val_ds)
    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    val_auc    = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)

    print(f"Epoch {epoch+1:2d} | train={train_loss:.4f} val={val_loss:.4f} AUC={val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        patience_count = 0
        torch.save({"model_state": model.state_dict(), "epoch": epoch+1, "auc": val_auc},
                   f"{SAVE_DIR}/best.pt")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stopping (patience={PATIENCE})")
            break

    scheduler.step()

print(f"\nMeilleure AUC val : {best_auc:.4f}")

Epoch  1 | train=0.2554 val=0.2669 AUC=0.9899
Epoch  2 | train=0.1586 val=0.5771 AUC=0.9734
Epoch  3 | train=0.1055 val=0.1370 AUC=0.9968
Epoch  4 | train=0.0517 val=0.1380 AUC=0.9980
Epoch  5 | train=0.0368 val=0.1194 AUC=0.9980
Epoch  6 | train=0.1223 val=0.3537 AUC=0.9873
Epoch  7 | train=0.1072 val=0.1999 AUC=0.9951
Epoch  8 | train=0.0814 val=0.2246 AUC=0.9954
Early stopping (patience=3)

Meilleure AUC val : 0.9980


### Ablation — sensibilité au taux de dropout

In [6]:
ABLATION_DIR = "runs_mc_dropout_ablation"
os.makedirs(ABLATION_DIR, exist_ok=True)

def train_mcd_variant(dropout_rate, epochs=EPOCHS, save_path=None):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    m = MCDropoutMultimodalNet(dropout=dropout_rate).to(device)

    ckpt_init = torch.load(CNN_CKPT, map_location="cpu", weights_only=False)
    state     = ckpt_init.get("model_state", ckpt_init)
    filtered  = {k: v for k, v in state.items() if not k.startswith("fc.")}
    m.cnn.load_state_dict(filtered, strict=False)

    trainable = [p for p in m.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    criterion = nn.CrossEntropyLoss()

    best_auc, patience_count = 0.0, 0

    for epoch in range(epochs):
        m.train()
        for imgs, clins, labels in train_loader:
            imgs, clins, labels = imgs.to(device), clins.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(m(imgs, clins), labels)
            loss.backward()
            optimizer.step()

        m.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for imgs, clins, labels in val_loader:
                imgs, clins = imgs.to(device), clins.to(device)
                all_probs.append(torch.softmax(m(imgs, clins), dim=1).cpu().numpy())
                all_labels.append(labels.numpy())
        all_probs  = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        val_auc    = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")

        print(f"  [dropout={dropout_rate}] epoch {epoch+1:2d}  val_AUC={val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc; patience_count = 0
            if save_path:
                torch.save({"model_state": m.state_dict(), "epoch": epoch+1,
                            "auc": val_auc, "dropout": dropout_rate}, save_path)
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"  Early stopping (patience={PATIENCE})")
                break
        scheduler.step()

    return best_auc


for rate in [0.2, 0.5]:
    print(f"\n=== Entraînement MC Dropout dropout={rate} ===")
    save_path = f"{ABLATION_DIR}/mcd_dropout_{rate}.pt"
    train_mcd_variant(rate, save_path=save_path)
    print(f"✓ Sauvegardé : {save_path}")


=== Entraînement MC Dropout dropout=0.2 ===
  [dropout=0.2] epoch  1  val_AUC=0.9799
  [dropout=0.2] epoch  2  val_AUC=0.9928
  [dropout=0.2] epoch  3  val_AUC=0.9963
  [dropout=0.2] epoch  4  val_AUC=0.9981
  [dropout=0.2] epoch  5  val_AUC=0.9984
  [dropout=0.2] epoch  6  val_AUC=0.9934
  [dropout=0.2] epoch  7  val_AUC=0.9956
  [dropout=0.2] epoch  8  val_AUC=0.9976
  Early stopping (patience=3)
✓ Sauvegardé : runs_mc_dropout_ablation/mcd_dropout_0.2.pt

=== Entraînement MC Dropout dropout=0.5 ===
  [dropout=0.5] epoch  1  val_AUC=0.9649
  [dropout=0.5] epoch  2  val_AUC=0.9951
  [dropout=0.5] epoch  3  val_AUC=0.9980
  [dropout=0.5] epoch  4  val_AUC=0.9980
  [dropout=0.5] epoch  5  val_AUC=0.9983
  [dropout=0.5] epoch  6  val_AUC=0.9967
  [dropout=0.5] epoch  7  val_AUC=0.9923
  [dropout=0.5] epoch  8  val_AUC=0.9972
  Early stopping (patience=3)
✓ Sauvegardé : runs_mc_dropout_ablation/mcd_dropout_0.5.pt


### Inférence MC Dropout sur le test

In [7]:
ckpt = torch.load(f"{SAVE_DIR}/best.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state"])

def mc_dropout_inference(loader, n_passes=MC_PASSES):
    model.eval()
    # garder le dropout actif à l'inférence pour simuler plusieurs réseaux
    model.enable_mc_dropout()

    all_means, all_stds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, clins, labels in loader:
            imgs, clins = imgs.to(device), clins.to(device)
            passes = torch.stack([
                torch.softmax(model(imgs, clins), dim=1)
                for _ in range(n_passes)
            ])
            all_means.append(passes.mean(dim=0).cpu().numpy())
            all_stds.append(passes.std(dim=0).cpu().numpy())
            all_labels.append(labels.numpy())

    return (np.concatenate(all_means),
            np.concatenate(all_stds),
            np.concatenate(all_labels))

means_mcd, stds_mcd, labels_mcd = mc_dropout_inference(test_loader)
preds_mcd = means_mcd.argmax(axis=1)

auc_mcd     = roc_auc_score(labels_mcd, means_mcd, multi_class="ovr", average="macro")
balacc_mcd  = balanced_accuracy_score(labels_mcd, preds_mcd)

confs_mcd    = means_mcd.max(axis=1)
correct_mcd  = (preds_mcd == labels_mcd).astype(float)
M = 10
bin_edges = np.linspace(0, 1, M+1)
ECE_mcd = 0.0
for i in range(M):
    mask = (confs_mcd >= bin_edges[i]) & (confs_mcd < bin_edges[i+1])
    if mask.sum() > 0:
        ECE_mcd += mask.sum()/len(confs_mcd) * abs(correct_mcd[mask].mean() - confs_mcd[mask].mean())

print(f"Performance MC Dropout (test Epic & CSCR)")
print(f"  AUC               : {auc_mcd:.4f}")
print(f"  Balanced accuracy : {balacc_mcd:.4f}")
print(f"  ECE               : {ECE_mcd:.4f}")

Performance MC Dropout (test Epic & CSCR)
  AUC               : 0.9967
  Balanced accuracy : 0.9750
  ECE               : 0.0116


### Ratio σ_incorrect / σ_correct

In [8]:
sigma_max_mcd = stds_mcd.max(axis=1)
correct_mask   = (preds_mcd == labels_mcd)
incorrect_mask = ~correct_mask

sigma_correct   = sigma_max_mcd[correct_mask].mean()
sigma_incorrect = sigma_max_mcd[incorrect_mask].mean()
ratio_global    = sigma_incorrect / sigma_correct

print(f"σ correct   : {sigma_correct:.4f}")
print(f"σ incorrect : {sigma_incorrect:.4f}")
print(f"Ratio global σ_incorrect / σ_correct : {ratio_global:.2f}")

print("\nPar classe :")
for i, cls in enumerate(CLASSES):
    cls_mask    = (labels_mcd == i)
    cls_correct = sigma_max_mcd[cls_mask & correct_mask].mean()
    cls_inc     = sigma_max_mcd[cls_mask & incorrect_mask]
    if len(cls_inc) > 0:
        print(f"  {cls:12s} ratio = {cls_inc.mean()/cls_correct:.2f}  ({len(cls_inc)} erreurs)")
    else:
        print(f"  {cls:12s} pas d'erreurs")

σ correct   : 0.0080
σ incorrect : 0.0728
Ratio global σ_incorrect / σ_correct : 9.10

Par classe :
  glioma       ratio = 5.71  (39 erreurs)
  meningioma   ratio = 5.96  (21 erreurs)
  notumor      ratio = 88.01  (1 erreurs)
  pituitary    ratio = 20.76  (5 erreurs)


### Tableau comparatif final - torchbnn vs MC Dropout

In [9]:
torchbnn_results = {
    "AUC":           0.995,
    "BalancedAcc":   0.964,
    "ECE":           0.0184,
    "Ratio_sigma":   7.41,
}

mcd_results = {
    "AUC":           auc_mcd,
    "BalancedAcc":   balacc_mcd,
    "ECE":           ECE_mcd,
    "Ratio_sigma":   ratio_global,
}

print(f"\n{'Métrique':<25} {'torchbnn':>12} {'MC Dropout':>12}")
print("-" * 51)
for k in torchbnn_results:
    fmt = ".2f" if k == "Ratio_sigma" else ".4f"
    print(f"{k:<25} {torchbnn_results[k]:>12{fmt}} {mcd_results[k]:>12{fmt}}")


Métrique                      torchbnn   MC Dropout
---------------------------------------------------
AUC                             0.9950       0.9967
BalancedAcc                     0.9640       0.9750
ECE                             0.0184       0.0116
Ratio_sigma                       7.41         9.10
